# ANLP Project — Kaggle Runner

Runs the football commentary summarization pipeline on a Kaggle GPU.

**Before running:** in the right sidebar set
- *Accelerator* → **GPU T4 x2** (or any GPU)
- *Internet* → **On**
- *Persistence* → optional

Each cell skips work if outputs already exist, so you can re-run safely.

## 1. Clone repo (dataset is bundled in branch `setup/local-run`)

In [ ]:
import os
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!ls

## 2. Install dependencies (skip what's already on Kaggle)

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

## 3. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi -L

## 4. Smoke test — 1 match, FLAN-T5 zero-shot chunk

Should take 2–10 minutes. If this works, the rest will too.

In [ ]:
!python scripts/run_prompting.py --model flan --strategy chunk --prompt zero --limit 1

## 5. (Optional) Run all 6 prompting conditions on full test set

**Heavy** — likely 4–8h total. Comment out conditions you don't want, or use `--limit 3` to keep it short.

In [ ]:
# Delete the previous limited run before doing a full run, or it'll be skipped
!rm -f outputs/predictions/flan_chunk_zero.json
!python scripts/run_prompting.py --model flan --strategy chunk --prompt zero
# !python scripts/run_prompting.py --model flan --strategy chunk --prompt few
# !python scripts/run_prompting.py --model flan --strategy chunk --prompt cot
# !python scripts/run_prompting.py --model led  --strategy long  --prompt zero
# !python scripts/run_prompting.py --model led  --strategy long  --prompt few
# !python scripts/run_prompting.py --model led  --strategy long  --prompt cot

## 6. Evaluate — ROUGE only (fast) or with BERTScore (slower)

In [ ]:
# BERTScore is buggy on the current Kaggle transformers/bert_score combo (OverflowError).
# Use --no-bertscore for ROUGE-only metrics.
!python scripts/run_evaluation.py --no-bertscore

## 7. Inspect results

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:600]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)

## 8. (Optional) Download outputs locally

After running, files in `/kaggle/working/ANLP_Project/outputs/` are saved as Notebook Output. Download them from the Kaggle UI → "Output" tab on the right.